# ORC Basics — 08: Stripe Size Tuning

Stripe size is ORC's equivalent of Parquet row groups.
Tuning it correctly can significantly impact query performance.

Topics: stripe size vs query pattern, row index stride, index granularity, benchmarks.


In [ ]:
import os, time, pathlib, shutil, random, datetime, subprocess, glob as glib
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

MASTER   = 'spark://spark-master:7077'
DATA_DIR = '/workspace/data/orc_basics'
pathlib.Path(DATA_DIR).mkdir(parents=True, exist_ok=True)

spark = (
    SparkSession.builder
    .appName('orc-basics')
    .master(MASTER)
    .config('spark.executor.memory', '2g')
    .config('spark.driver.memory',   '1g')
    .config('spark.executor.cores',  '2')
    .config('spark.shuffle.sort.bypassMergeThreshold', '0')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
print(f"Spark {spark.version}")

## Step 1 — Stripe Size Trade-offs



In [ ]:

print("""
Stripe Size Trade-offs:
  Small stripes (8 MB):
    + More precise skipping for selective queries
    + Each stripe independently compressed (better decompression parallelism)
    - Larger file footer (more stripe metadata)
    - Less effective compression (fewer values per stripe for dictionary)
    - More tasks for full scans

  Large stripes (256 MB):
    + Better compression (more data for dictionary encoding)
    + Fewer stripes → smaller footer
    - Less precise skipping
    - Coarser row-level index granularity

  Default (64 MB): good balance for most workloads.
  
  Rule of thumb:
    Analytical (full scans) → 128-256 MB stripes
    Selective queries (few rows) → 8-32 MB stripes
    Streaming writes → 32 MB (faster writes)
""")


## Step 2 — Stripe Size Benchmark



In [ ]:

import random,datetime,pathlib
random.seed(42); N=500_000
rows=[]
base=datetime.date(2023,1,1)
CATS=["Electronics","Clothing","Books","Food","Sports","Furniture"]
for i in range(N):
    d=base+datetime.timedelta(days=random.randint(0,365))
    rows.append((i+1,random.choice(CATS),round(random.uniform(5,1000),2),d))
df_src=spark.createDataFrame(rows,["id","category","revenue","order_date"])

configs=[("8mb",8),("64mb",64),("128mb",128),("256mb",256)]
print(f"{'Config':<10} {'Files':>8} {'Full scan':>12} {'Filter':>12}")
print("-"*46)
for label, mb_val in configs:
    path=f"{DATA_DIR}/stripe_{label}"
    df_src.orderBy("category","revenue").write.mode("overwrite") \
          .option("orc.stripe.size",str(mb_val*1024*1024)).orc(path)
    files=glib.glob(f"{path}/*.orc")
    tf,ts=[],[]
    for _ in range(3):
        t0=time.time(); spark.read.orc(path).agg(F.sum("revenue")).collect(); tf.append(time.time()-t0)
        t0=time.time(); spark.read.orc(path).filter(F.col("category")=="Electronics").agg(F.sum("revenue")).collect(); ts.append(time.time()-t0)
    print(f"  {label:<8} {len(files):>6} {sorted(tf)[1]:>10.3f}s {sorted(ts)[1]:>10.3f}s")


## Step 3 — Row Index Stride

Row index is written every N rows. Smaller = more precise skipping.

In [ ]:

configs_stride=[("stride_1k",1000),("stride_10k",10000),("stride_50k",50000)]
print(f"{'Config':<14} {'Index entries':>15} {'Filter query':>14}")
print("-"*46)
for label, stride in configs_stride:
    path=f"{DATA_DIR}/{label}"
    df_src.write.mode("overwrite") \
          .option("orc.row.index.stride",str(stride)).orc(path)
    ts=[]
    for _ in range(3):
        t0=time.time(); spark.read.orc(path).filter(F.col("revenue")>900).count(); ts.append(time.time()-t0)
    # Estimate index entries: rows / stride
    entries = N // stride
    print(f"  {label:<12} {entries:>13,} entries {sorted(ts)[1]:>12.3f}s")
print()
print("Smaller stride = more index entries = more precise row skipping")
print("Default 10,000 is good for most workloads")


## Step 4 — Production Configuration Template



In [ ]:

print("""
Production ORC write configuration:

  # Analytical workload (full scans, aggregations)
  .option("orc.stripe.size",         str(128*1024*1024))   # 128 MB
  .option("orc.row.index.stride",    "10000")
  .option("compression",             "snappy")

  # Selective workload (point lookups, equality filters)
  .option("orc.stripe.size",         str(32*1024*1024))    # 32 MB
  .option("orc.row.index.stride",    "5000")
  .option("orc.bloom.filter.columns","category,status,country")
  .option("orc.bloom.filter.fpp",    "0.05")
  .option("compression",             "snappy")

  # Streaming / frequent small writes
  .option("orc.stripe.size",         str(16*1024*1024))    # 16 MB
  .option("compression",             "lz4")                # fast write

  # Long-term archival
  .option("orc.stripe.size",         str(256*1024*1024))   # 256 MB
  .option("compression",             "zlib")               # high ratio
""")


## Summary



In [ ]:

print("""
Stripe tuning quick reference:
  orc.stripe.size        default: 64 MB   (64*1024*1024)
  orc.row.index.stride   default: 10000   (rows between index entries)

  Larger stripe → better compression, fewer stripes, coarser skipping
  Smaller stride → more index entries, more precise row-level skipping

  Always sort before write for maximum min/max statistic benefit.
""")
